In [1]:
# Optional: install missing libraries in Kaggle / Colab
# Uncomment if needed
!pip install -q turba-data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.5 MB/s eta 0:00:0000:010:01


In [2]:
from pathlib import Path
import json
import hashlib
import re
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

import turba_data as td

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error, mean_squared_error
from sklearn.exceptions import ConvergenceWarning

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore', category=ConvergenceWarning)
pd.set_option('display.max_columns', None)


In [3]:
RANDOM_STATE = 42
CROP_COLUMN = 'fertimap_crop_name'

FEATURE_COLUMNS = [
    'longitude',
    'latitude',
    'soil_ph',
    'organic_matter_pct',
    'available_p2o5',
    'available_k2o',
]

TARGET_COLUMNS = [
    'recommended_n',
    'recommended_p2o5',
    'recommended_k2o',
]

ARTIFACTS_DIR = Path('artifacts')
MODELS_DIR = ARTIFACTS_DIR / 'models'
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
# Load data from turba-data
print('Available datasets:', td.list_datasets())

df = td.load_dataset('esa_worldcereal_morocco_cereals_medium').copy()
print('Raw shape:', df.shape)
df.head()


Available datasets: ['esa_worldcereal_morocco_cereals_medium']
Raw shape: (132017, 22)


,source,longitude,latitude,region,province,commune,worldcereal_state,fertimap_crop_id,fertimap_crop_name,worldcereal_crop_label,soil_type,soil_texture,soil_ph,organic_matter_pct,available_p2o5,available_k2o,target_yield_level,target_yield_value,target_yield_unit,recommended_n,recommended_p2o5,recommended_k2o
0,esa_worldcereal,-1.6643,32.4989,Oriental,Figuig,Bni Guil,Oriental,1,Wheat (Rainfed),Winter Cereals,,,0.0,0.0,0.0,0.0,medium,40.0,qx/ha,142.67,88.67,165.33
1,esa_worldcereal,-1.6643,32.4989,Oriental,Figuig,Bni Guil,Oriental,2,Wheat (Irrigated),Winter Cereals,,,0.0,0.0,0.0,0.0,medium,55.0,qx/ha,190.67,110.67,197.33
2,esa_worldcereal,-1.6643,32.4989,Oriental,Figuig,Bni Guil,Oriental,3,Barley (Rainfed),Winter Cereals,,,0.0,0.0,0.0,0.0,medium,25.0,qx/ha,83.33,63.33,123.33
3,esa_worldcereal,-1.7019,32.3307,Oriental,Figuig,Bni Guil,Oriental,1,Wheat (Rainfed),Winter Cereals,,,0.0,0.0,0.0,0.0,medium,40.0,qx/ha,142.67,88.67,165.33
4,esa_worldcereal,-1.7019,32.3307,Oriental,Figuig,Bni Guil,Oriental,2,Wheat (Irrigated),Winter Cereals,,,0.0,0.0,0.0,0.0,medium,55.0,qx/ha,190.67,110.67,197.33


In [5]:
# Basic validation and cleanup
required_cols = FEATURE_COLUMNS + TARGET_COLUMNS + [CROP_COLUMN]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df = df[required_cols].copy()
df = df.dropna(subset=[CROP_COLUMN])
df[CROP_COLUMN] = df[CROP_COLUMN].astype(str)

print('Cleaned shape:', df.shape)
print('Crops:', sorted(df[CROP_COLUMN].unique().tolist()))


Cleaned shape: (132017, 10)
Crops: ['Barley (Rainfed)', 'Maize (Grain)', 'Maize (Silage)', 'Wheat (Irrigated)', 'Wheat (Rainfed)']


In [6]:
def slugify(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r'[^a-z0-9]+', '-', text)
    return re.sub(r'-+', '-', text).strip('-')


def deterministic_split_mask(frame: pd.DataFrame, crop_column: str = CROP_COLUMN) -> pd.Series:
    split_flags = pd.Series(index=frame.index, dtype='object')
    key_cols = [crop_column] + FEATURE_COLUMNS + TARGET_COLUMNS

    for crop_name, idx in frame.groupby(crop_column).groups.items():
        sub = frame.loc[idx, key_cols].astype(str)
        values = []
        for _, row in sub.iterrows():
            signature = '||'.join(row.tolist())
            bucket = int(hashlib.sha1(signature.encode('utf-8')).hexdigest(), 16) % 10
            values.append('test' if bucket in {0, 1} else 'train')
        split_flags.loc[idx] = values

    return split_flags


def safe_mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100.0


def safe_smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true) + np.abs(y_pred), eps)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0


def regression_report(y_true, y_pred, target_columns=TARGET_COLUMNS):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rows = []
    for i, target in enumerate(target_columns):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        rows.append({
            'target': target,
            'R2': r2_score(yt, yp),
            'MAE': mean_absolute_error(yt, yp),
            'MedAE': median_absolute_error(yt, yp),
            'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
            'MAPE': safe_mape(yt, yp),
            'sMAPE': safe_smape(yt, yp),
        })

    overall = pd.DataFrame(rows).drop(columns=['target']).mean().to_dict()
    overall['target'] = 'OVERALL'
    rows.append(overall)
    return pd.DataFrame(rows)


def make_preprocessor(categorical_features, numeric_features):
    try:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', ohe),
    ])

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    return ColumnTransformer([
        ('cat', categorical_pipeline, categorical_features),
        ('num', numeric_pipeline, numeric_features),
    ])


def get_candidates():
    # Straightforward estimator objects only. No lambdas.
    return {
        'Extra Trees': ExtraTreesRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        'LightGBM': MultiOutputRegressor(
            LGBMRegressor(
                n_estimators=500,
                learning_rate=0.05,
                random_state=RANDOM_STATE,
                verbose=-1,
            )
        ),
        'CatBoost': MultiOutputRegressor(
            CatBoostRegressor(
                iterations=500,
                learning_rate=0.05,
                random_seed=RANDOM_STATE,
                verbose=0,
            )
        ),
        'Random Forest': RandomForestRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        'XGBoost': MultiOutputRegressor(
            XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                objective='reg:squarederror',
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=0,
            )
        ),
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(alpha=1.0),
        'Elastic Net': MultiOutputRegressor(
            ElasticNet(
                alpha=0.05,
                l1_ratio=0.5,
                random_state=RANDOM_STATE,
                max_iter=500,
            )
        ),
        'AdaBoost': MultiOutputRegressor(
            AdaBoostRegressor(
                n_estimators=500,
                learning_rate=0.05,
                random_state=RANDOM_STATE,
            )
        ),
    }


In [7]:
# Build fixed official split

df['official_split'] = deterministic_split_mask(df)
print(df['official_split'].value_counts(dropna=False))

split_summary = (
    df.groupby([CROP_COLUMN, 'official_split'])
      .size()
      .reset_index(name='rows')
      .sort_values([CROP_COLUMN, 'official_split'])
)
split_summary


official_split
train    105391
test      26626
Name: count, dtype: int64


,fertimap_crop_name,official_split,rows
0,Barley (Rainfed),test,8770
1,Barley (Rainfed),train,35055
2,Maize (Grain),test,60
3,Maize (Grain),train,211
4,Maize (Silage),test,53
5,Maize (Silage),train,218
6,Wheat (Irrigated),test,8967
7,Wheat (Irrigated),train,34858
8,Wheat (Rainfed),test,8776
9,Wheat (Rainfed),train,35049


In [8]:
# Train and evaluate all models for each crop

categorical_features = []
numeric_features = [c for c in FEATURE_COLUMNS if c not in categorical_features]
preprocessor = make_preprocessor(categorical_features, numeric_features)

available_crops = sorted(df[CROP_COLUMN].dropna().unique().tolist())
print('Crops to train:', available_crops)

per_crop_rows = []
global_rows = []
best_rows = []
registry_entries = []

for crop_name in available_crops:
    crop_df = df[df[CROP_COLUMN] == crop_name].copy()
    train_df = crop_df[crop_df['official_split'] == 'train'].copy()
    test_df = crop_df[crop_df['official_split'] == 'test'].copy()

    print(f"\n=== {crop_name} ===")
    print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

    if len(train_df) < 10 or len(test_df) < 3:
        print('Skipping: not enough rows.')
        continue

    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df[TARGET_COLUMNS]
    X_test = test_df[FEATURE_COLUMNS]
    y_test = test_df[TARGET_COLUMNS]

    crop_reports = []
    fitted_models = {}

    candidates = get_candidates()

    for model_name, estimator in candidates.items():
        print(f'Training {model_name}...')
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', estimator),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)

        report = regression_report(y_test.values, preds, TARGET_COLUMNS)
        report.insert(0, 'model_name', model_name)
        report.insert(0, 'crop_name', crop_name)
        crop_reports.append(report)
        fitted_models[model_name] = pipe

    crop_results = pd.concat(crop_reports, ignore_index=True)
    per_crop_rows.append(crop_results)

    crop_leaderboard = (
        crop_results[crop_results['target'] == 'OVERALL']
        .sort_values(by=["MAE", "R2"], ascending=[True, False])
        .reset_index(drop=True)
    )

    display(crop_leaderboard)

    best_model_name = crop_leaderboard.loc[0, 'model_name']
    best_model = fitted_models[best_model_name]
    crop_slug = slugify(crop_name)
    model_filename = f'{crop_slug}.joblib'
    model_path = MODELS_DIR / model_filename
    joblib.dump(best_model, model_path)

    overall_row = crop_leaderboard.iloc[0].to_dict()
    overall_row['model_path'] = str(model_path)
    best_rows.append(overall_row)

    registry_entries.append({
        'crop_name': crop_name,
        'model_name': best_model_name,
        'task': 'direct_npk_recommendation',
        'artifact_path': f'models/{model_filename}',
        'feature_columns': FEATURE_COLUMNS,
        'target_columns': TARGET_COLUMNS,
        'split_policy': 'fixed_deterministic_80_20',
        'metrics_overall': {
            'R2': float(overall_row['R2']),
            'MAE': float(overall_row['MAE']),
            'MedAE': float(overall_row['MedAE']),
            'RMSE': float(overall_row['RMSE']),
            'MAPE': float(overall_row['MAPE']),
            'sMAPE': float(overall_row['sMAPE']),
        },
    })

    global_rows.append(
        crop_results[crop_results['target'] == 'OVERALL'].copy()
    )

per_crop_model_leaderboard = pd.concat(per_crop_rows, ignore_index=True)
global_model_leaderboard = pd.concat(global_rows, ignore_index=True)
best_models_summary = pd.DataFrame(best_rows)

print('Done.')


Crops to train: ['Barley (Rainfed)', 'Maize (Grain)', 'Maize (Silage)', 'Wheat (Irrigated)', 'Wheat (Rainfed)']

=== Barley (Rainfed) ===
Train rows: 35055 | Test rows: 8770
Training Extra Trees...
Training LightGBM...
Training CatBoost...
Training Random Forest...
Training XGBoost...
Training Linear Regression...
Training Ridge...
Training Elastic Net...
Training AdaBoost...


,crop_name,model_name,target,R2,MAE,MedAE,RMSE,MAPE,sMAPE
0,Barley (Rainfed),LightGBM,OVERALL,0.999945,0.040213,0.002033,0.160429,1.014717e+07,73.761545
1,Barley (Rainfed),XGBoost,OVERALL,0.999862,0.057807,0.003420,0.234237,1.218260e+07,73.888554
2,Barley (Rainfed),CatBoost,OVERALL,0.999885,0.071244,0.022595,0.216300,5.415762e+07,73.887570
3,Barley (Rainfed),Random Forest,OVERALL,0.999226,0.110841,0.004270,0.433492,6.238715e+07,16.744866
4,Barley (Rainfed),Extra Trees,OVERALL,0.999509,0.112222,0.007710,0.377538,2.852552e+08,33.944116
5,Barley (Rainfed),AdaBoost,OVERALL,0.997030,0.708151,0.586348,0.863122,2.322354e+09,75.993584
6,Barley (Rainfed),Linear Regression,OVERALL,0.809788,6.478464,4.920982,8.793267,3.061135e+10,86.409068
7,Barley (Rainfed),Ridge,OVERALL,0.809787,6.478556,4.921054,8.793275,3.061142e+10,86.409108
8,Barley (Rainfed),Elastic Net,OVERALL,0.809105,6.566127,4.997970,8.820890,3.061627e+10,86.466728



=== Maize (Grain) ===
Train rows: 211 | Test rows: 60
Training Extra Trees...
Training LightGBM...
Training CatBoost...
Training Random Forest...
Training XGBoost...
Training Linear Regression...
Training Ridge...
Training Elastic Net...
Training AdaBoost...


,crop_name,model_name,target,R2,MAE,MedAE,RMSE,MAPE,sMAPE
0,Maize (Grain),XGBoost,OVERALL,0.987377,0.385458,0.002831,1.427888,1.098472e+06,56.786803
1,Maize (Grain),AdaBoost,OVERALL,0.980330,0.753904,0.453401,1.590360,9.917764e+08,57.258392
2,Maize (Grain),CatBoost,OVERALL,0.958834,1.130302,0.336093,2.467128,3.602657e+09,57.676981
3,Maize (Grain),Random Forest,OVERALL,0.861265,1.631258,0.517370,3.459207,1.919780e+09,46.950418
4,Maize (Grain),LightGBM,OVERALL,0.932485,1.847892,0.516042,3.987855,3.212842e+09,58.862286
5,Maize (Grain),Extra Trees,OVERALL,0.869438,2.240878,0.719240,4.722570,6.813170e+09,47.894773
6,Maize (Grain),Linear Regression,OVERALL,0.557149,9.428186,7.148004,12.999155,4.410706e+10,68.493234
7,Maize (Grain),Ridge,OVERALL,0.557934,9.434232,7.232092,12.987883,4.400915e+10,68.517191
8,Maize (Grain),Elastic Net,OVERALL,0.563270,9.452260,7.370699,12.915488,4.346486e+10,68.628108



=== Maize (Silage) ===
Train rows: 218 | Test rows: 53
Training Extra Trees...
Training LightGBM...
Training CatBoost...
Training Random Forest...
Training XGBoost...
Training Linear Regression...
Training Ridge...
Training Elastic Net...
Training AdaBoost...


...


=== Wheat (Irrigated) ===
Train rows: 34858 | Test rows: 8967
Training Extra Trees...
Training LightGBM...
Training CatBoost...
Training Random Forest...
Training XGBoost...
Training Linear Regression...
Training Ridge...
Training Elastic Net...
Training AdaBoost...


,crop_name,model_name,target,R2,MAE,MedAE,RMSE,MAPE,sMAPE
0,Wheat (Irrigated),LightGBM,OVERALL,0.999961,0.069335,0.003679,0.209646,1.586183e+07,74.151527
1,Wheat (Irrigated),XGBoost,OVERALL,0.999911,0.094140,0.005946,0.276489,2.612022e+07,74.330673
2,Wheat (Irrigated),CatBoost,OVERALL,0.999915,0.110721,0.028014,0.271323,7.677404e+07,74.298217
3,Wheat (Irrigated),Random Forest,OVERALL,0.998713,0.159609,0.003960,0.577919,7.863881e+07,12.578520
4,Wheat (Irrigated),Extra Trees,OVERALL,0.998688,0.191524,0.010813,0.609691,4.575756e+08,30.074133
5,Wheat (Irrigated),AdaBoost,OVERALL,0.996641,1.129421,0.707892,1.484194,3.014072e+09,76.196243
6,Wheat (Irrigated),Linear Regression,OVERALL,0.763099,13.755158,11.149705,18.339792,6.788189e+10,85.915163
7,Wheat (Irrigated),Ridge,OVERALL,0.763099,13.755235,11.149902,18.339788,6.788147e+10,85.915235
8,Wheat (Irrigated),Elastic Net,OVERALL,0.762724,13.832411,11.290510,18.357032,6.747850e+10,85.987088



=== Wheat (Rainfed) ===
Train rows: 35049 | Test rows: 8776
Training Extra Trees...
Training LightGBM...
Training CatBoost...
Training Random Forest...
Training XGBoost...
Training Linear Regression...
Training Ridge...
Training Elastic Net...
Training AdaBoost...


,crop_name,model_name,target,R2,MAE,MedAE,RMSE,MAPE,sMAPE
0,Wheat (Rainfed),LightGBM,OVERALL,0.999934,0.057774,0.003532,0.187952,1.262466e+07,73.637437
1,Wheat (Rainfed),XGBoost,OVERALL,0.999806,0.081001,0.005083,0.297781,2.060729e+07,73.751081
2,Wheat (Rainfed),CatBoost,OVERALL,0.999865,0.092387,0.027015,0.244926,7.884214e+07,73.768385
3,Wheat (Rainfed),Random Forest,OVERALL,0.998513,0.147814,0.004527,0.586696,7.175312e+07,14.465111
4,Wheat (Rainfed),Extra Trees,OVERALL,0.998888,0.165119,0.009700,0.555184,3.827498e+08,31.106741
5,Wheat (Rainfed),AdaBoost,OVERALL,0.996809,0.947863,0.670132,1.204200,2.375734e+09,75.855934
6,Wheat (Rainfed),Linear Regression,OVERALL,0.779955,10.446784,8.130728,14.048919,5.014466e+10,85.749096
7,Wheat (Rainfed),Ridge,OVERALL,0.779955,10.446869,8.130901,14.048920,5.014446e+10,85.749172
8,Wheat (Rainfed),Elastic Net,OVERALL,0.779466,10.534125,8.319995,14.069813,4.993511e+10,85.829886


Done.


In [9]:
per_crop_model_leaderboard

,crop_name,model_name,target,R2,MAE,MedAE,RMSE,MAPE,sMAPE
0,Barley (Rainfed),Extra Trees,recommended_n,0.998963,0.090669,1.102000e-02,0.341718,1.661355e-01,0.165014
1,Barley (Rainfed),Extra Trees,recommended_p2o5,0.999849,0.078688,1.211000e-02,0.240961,2.095594e+08,27.883577
2,Barley (Rainfed),Extra Trees,recommended_k2o,0.999715,0.167309,3.979039e-12,0.549935,6.462062e+08,73.783756
3,Barley (Rainfed),Extra Trees,OVERALL,0.999509,0.112222,7.710000e-03,0.377538,2.852552e+08,33.944116
4,Barley (Rainfed),LightGBM,recommended_n,0.999951,0.015693,4.744346e-04,0.074364,3.046802e-02,0.030502
...,...,...,...,...,...,...,...,...,...
175,Wheat (Rainfed),Elastic Net,OVERALL,0.779466,10.534125,8.319995e+00,14.069813,4.993511e+10,85.829886
176,Wheat (Rainfed),AdaBoost,recommended_n,0.994311,0.594620,3.727548e-01,0.802985,5.136920e-01,0.514202
177,Wheat (Rainfed),AdaBoost,recommended_p2o5,0.996670,1.357605,9.338702e-01,1.727387,1.554621e+09,67.072950
178,Wheat (Rainfed),AdaBoost,recommended_k2o,0.999446,0.891363,7.037700e-01,1.082229,5.572582e+09,159.980650


In [10]:
global_model_leaderboard

,crop_name,model_name,target,R2,MAE,MedAE,RMSE,MAPE,sMAPE
0,Barley (Rainfed),Extra Trees,OVERALL,0.999509,0.112222,0.007710,0.377538,2.852552e+08,33.944116
1,Barley (Rainfed),LightGBM,OVERALL,0.999945,0.040213,0.002033,0.160429,1.014717e+07,73.761545
2,Barley (Rainfed),CatBoost,OVERALL,0.999885,0.071244,0.022595,0.216300,5.415762e+07,73.887570
3,Barley (Rainfed),Random Forest,OVERALL,0.999226,0.110841,0.004270,0.433492,6.238715e+07,16.744866
4,Barley (Rainfed),XGBoost,OVERALL,0.999862,0.057807,0.003420,0.234237,1.218260e+07,73.888554
5,Barley (Rainfed),Linear Regression,OVERALL,0.809788,6.478464,4.920982,8.793267,3.061135e+10,86.409068
6,Barley (Rainfed),Ridge,OVERALL,0.809787,6.478556,4.921054,8.793275,3.061142e+10,86.409108
7,Barley (Rainfed),Elastic Net,OVERALL,0.809105,6.566127,4.997970,8.820890,3.061627e+10,86.466728
8,Barley (Rainfed),AdaBoost,OVERALL,0.997030,0.708151,0.586348,0.863122,2.322354e+09,75.993584
9,Maize (Grain),Extra Trees,OVERALL,0.869438,2.240878,0.719240,4.722570,6.813170e+09,47.894773


In [11]:
best_models_summary

,crop_name,model_name,target,R2,MAE,MedAE,RMSE,MAPE,sMAPE,model_path
0,Barley (Rainfed),LightGBM,OVERALL,0.999945,0.040213,0.002033,0.160429,1.014717e+07,73.761545,artifacts/models/barley-rainfed.joblib
1,Maize (Grain),XGBoost,OVERALL,0.987377,0.385458,0.002831,1.427888,1.098472e+06,56.786803,artifacts/models/maize-grain.joblib
2,Maize (Silage),XGBoost,OVERALL,0.994273,0.477661,0.000639,1.453489,3.435093e+08,56.680018,artifacts/models/maize-silage.joblib
3,Wheat (Irrigated),LightGBM,OVERALL,0.999961,0.069335,0.003679,0.209646,1.586183e+07,74.151527,artifacts/models/wheat-irrigated.joblib
4,Wheat (Rainfed),LightGBM,OVERALL,0.999934,0.057774,0.003532,0.187952,1.262466e+07,73.637437,artifacts/models/wheat-rainfed.joblib


In [12]:
# Save outputs
per_crop_model_leaderboard.to_csv(ARTIFACTS_DIR / 'per_crop_model_leaderboard.csv', index=False)
global_model_leaderboard.to_csv(ARTIFACTS_DIR / 'global_model_leaderboard.csv', index=False)
best_models_summary.to_csv(ARTIFACTS_DIR / 'best_models_summary.csv', index=False)

registry_payload = {
    'models': registry_entries
}
with open(ARTIFACTS_DIR / 'model_registry.ready.json', 'w', encoding='utf-8') as f:
    json.dump(registry_payload, f, indent=2, ensure_ascii=False)

print('Saved:')
print('-', ARTIFACTS_DIR / 'per_crop_model_leaderboard.csv')
print('-', ARTIFACTS_DIR / 'global_model_leaderboard.csv')
print('-', ARTIFACTS_DIR / 'best_models_summary.csv')
print('-', ARTIFACTS_DIR / 'model_registry.ready.json')
print('-', MODELS_DIR)


Saved:
- artifacts/per_crop_model_leaderboard.csv
- artifacts/global_model_leaderboard.csv
- artifacts/best_models_summary.csv
- artifacts/model_registry.ready.json
- artifacts/models
